# Validation-only temporal-regularization selection

This notebook selects `lambda_TS` **without opening the final test set**.
It operates only on corrected candidate workbooks in `excel files/remediation/`
and uses the validation metrics stored in `svcj_params` together with
optimizer/stability/admissibility diagnostics.

The notebook does not edit the lock record automatically. After reviewing
the table and figure, update `docs/reproducibility/regularization_selection.md`
and commit that lock before running the final Chapter 3 notebook.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

NOTEBOOK_ROOT = Path.cwd().resolve()
if (NOTEBOOK_ROOT / "_lib").exists():
    NOTEBOOK_DIR = NOTEBOOK_ROOT
elif (NOTEBOOK_ROOT / "notebooks" / "_lib").exists():
    NOTEBOOK_DIR = NOTEBOOK_ROOT / "notebooks"
else:
    raise FileNotFoundError(f"Could not locate notebooks/_lib from {NOTEBOOK_ROOT}")

if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from _lib import chapter3_final as chapter3
from _lib import common

PATHS = common.locate_notebook_paths(NOTEBOOK_DIR)
REMEDIATION_EXCEL_DIR = PATHS.excel_dir / "remediation"
OUTPUT_DIR = PATHS.notebook_dir / "chapter3_outputs" / "remediation" / "regularization_selection"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FILE_PATTERN = "calibration_results_reg_*.xlsx"

## 1. Load corrected candidate workbooks

In [ ]:
files = chapter3.find_calibration_files([REMEDIATION_EXCEL_DIR], FILE_PATTERN)
[path.name for path in files]

## 2. Summarize validation performance and diagnostics

In [ ]:
reg_summary = chapter3.compute_regularisation_summary(
    files,
    OUTPUT_DIR,
    evaluation_split="validation",
)
reg_summary

## 3. Apply the pre-specified validation selection rule

In [ ]:
selected_reg = chapter3.select_regularisation_candidate(reg_summary)
print(f"Validation-selected lambda_TS = {selected_reg}")

In [ ]:
fig = chapter3.build_regularisation_tradeoff_figure(
    reg_summary,
    final_reg=selected_reg,
    output_dir=OUTPUT_DIR,
    evaluation_split="validation",
)
fig.show()

## 4. Lock before final reporting

Review the validation table/figure above. Then update and commit
`docs/reproducibility/regularization_selection.md` with `Status: LOCKED`,
the selected integer value, and the candidate-results commit SHA. Do not
run the final Chapter 3 reporting notebook before that commit exists.